In [2]:
import time
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import pandas as pd

# -----------------------------
# Inputs
# -----------------------------
shapefile_path = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\Hawaii\Hawaii_3_10.shp"
ghi_raster = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\Hawaii\Non-Arctic_Average_GHI_y2005_2024_6634.tif"
output_csv = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\Hawaii\Hawaii_GHI.csv"
log_file = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\Hawaii\Hawaii_GHI.txt"

# -----------------------------
# Logger (flush enabled)
# -----------------------------
def log(msg):
    print(msg, flush=True)
    with open(log_file, "a") as f:
        f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}\n")

# -----------------------------
# Start
# -----------------------------
log("Starting Hawaii GHI CSV export script.")

# -----------------------------
# Load Shapefile
# -----------------------------
log("Loading shapefile...")
gdf = gpd.read_file(shapefile_path)
total_polygons = len(gdf)
log(f"Loaded {total_polygons} polygons.")

# -----------------------------
# CRS Check
# -----------------------------
log("Checking CRS information...")

vector_crs = gdf.crs
log(f"Shapefile CRS: {vector_crs}")

with rasterio.open(ghi_raster) as src:
    raster_crs = src.crs

log(f"Raster CRS: {raster_crs}")

if vector_crs != raster_crs:
    log("WARNING: CRS mismatch detected!")
    log("Reproject shapefile to match raster before running for accurate results.")
else:
    log("CRS match confirmed.")

# -----------------------------
# Zonal Stats with Progress
# -----------------------------
log("Beginning zonal statistics computation...")

start_time = time.time()
ghi_means = []

for idx, row in gdf.iterrows():

    stat = zonal_stats(
        row.geometry,
        ghi_raster,
        stats=["mean"],
        all_touched=True,
        nodata=-9999
    )[0]["mean"]

    ghi_means.append(stat)

    # ---- Progress Update ----
    percent_complete = ((idx + 1) / total_polygons) * 100

    # Print every 1% or final row
    if (idx + 1) % max(1, total_polygons // 100) == 0 or (idx + 1) == total_polygons:
        elapsed = (time.time() - start_time) / 60
        log(f"Progress: {percent_complete:.1f}% | "
            f"{idx+1}/{total_polygons} polygons | "
            f"Elapsed: {elapsed:.2f} minutes")

# -----------------------------
# Build Output DataFrame
# -----------------------------
log("Building output dataframe...")

output_df = pd.DataFrame({
    "ROW_ID": gdf["ROW_ID"],
    "GHI_Mean": ghi_means
})

# -----------------------------
# Export CSV
# -----------------------------
output_df.to_csv(output_csv, index=False)
log(f"CSV successfully saved: {output_csv}")

total_time = (time.time() - start_time) / 60
log(f"Hawaii script completed successfully in {total_time:.2f} minutes.")

Starting Hawaii GHI CSV export script.
Loading shapefile...
Loaded 504 polygons.
Checking CRS information...
Shapefile CRS: EPSG:6634
Raster CRS: EPSG:6634
CRS match confirmed.
Beginning zonal statistics computation...
Progress: 1.0% | 5/504 polygons | Elapsed: 0.00 minutes
Progress: 2.0% | 10/504 polygons | Elapsed: 0.00 minutes
Progress: 3.0% | 15/504 polygons | Elapsed: 0.00 minutes
Progress: 4.0% | 20/504 polygons | Elapsed: 0.00 minutes
Progress: 5.0% | 25/504 polygons | Elapsed: 0.00 minutes
Progress: 6.0% | 30/504 polygons | Elapsed: 0.00 minutes
Progress: 6.9% | 35/504 polygons | Elapsed: 0.00 minutes
Progress: 7.9% | 40/504 polygons | Elapsed: 0.00 minutes
Progress: 8.9% | 45/504 polygons | Elapsed: 0.00 minutes
Progress: 9.9% | 50/504 polygons | Elapsed: 0.01 minutes
Progress: 10.9% | 55/504 polygons | Elapsed: 0.01 minutes
Progress: 11.9% | 60/504 polygons | Elapsed: 0.01 minutes
Progress: 12.9% | 65/504 polygons | Elapsed: 0.01 minutes
Progress: 13.9% | 70/504 polygons | El